In [0]:
from pyspark.sql.types import StructType

TABLE = "cat_adblh_dev.bronze.shipments_postgres_raw"

LANDING = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments"
)

CHECKPOINT_ROOT = (
    "abfss://checkpoints@stadblhdeveus2001.dfs.core.windows.net/"
)
CHECKPOINT_NAME = "q2_shipments_directory_listing_v1"
CHECKPOINT = CHECKPOINT_ROOT + CHECKPOINT_NAME

SOURCE_COLUMNS = [
    "shipment_id",
    "order_id",
    "shipment_status",
    "updated_at",
    "source_version",
]

# Preserve the actual source-field types before clearing Bronze.
existing = spark.table(TABLE)
SOURCE_SCHEMA = StructType([
    existing.schema[name] for name in SOURCE_COLUMNS
])

# Preserve the project's actual source-system label.
labels = [
    row["_source_system"]
    for row in existing.select("_source_system").distinct().collect()
]
if len(labels) != 1 or labels[0] is None:
    raise RuntimeError(
        "Expected one non-null source-system label before the reset."
    )

SOURCE_SYSTEM = labels[0]

# PROBLEM — Clearing Bronze while retaining completed progress
# could prevent already-processed files from refilling the table.
# Require an unused checkpoint for this deliberate rebuild.
checkpoint_entries = dbutils.fs.ls(CHECKPOINT_ROOT)

if any(
    item.name.rstrip("/") == CHECKPOINT_NAME
    for item in checkpoint_entries
):
    raise RuntimeError(
        "This checkpoint already exists. Stop before clearing anything."
    )

print("Source schema and source-system label captured.")
print(f"Checkpoint selected: {CHECKPOINT}")
print("No data has been changed.")

In [0]:
# Run once, after removing the five landed file copies.
# This clears table records, not the table definition or source files.
spark.sql(f"TRUNCATE TABLE {TABLE}")

print("Bronze records cleared. Upload the baseline, then run cell 3.")

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

# Each execution has its own identifier.
run_id = str(uuid4())

# PROBLEM — Manual delivery selection and last-filename tracking.
# Auto Loader reads the landing root and maintains file-processing state.
source_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useManagedFileEvents", "false")
    .option("cloudFiles.useNotifications", "false")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.partitionColumns", "")
    .schema(SOURCE_SCHEMA)
    .load(LANDING)
)

# Preserve source fields and record where each row came from.
bronze_stream = (
    source_stream.select(
        *SOURCE_COLUMNS,
        F.col("_metadata.file_path").alias("_source_file"),
    )
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn(
        "_delivery_id",
        F.regexp_extract(
            F.col("_source_file"),
            r"/delivery_id=([^/]+)/",
            1,
        ),
    )
    .withColumn("_run_id", F.lit(run_id))
    .withColumn("_ingested_at", F.current_timestamp())
)

# PROBLEM — Duplicate ingestion during normal restart.
# Reuse this checkpoint and target for every arrival and retry.
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)

# Wait until this execution finishes before reading the result.
query.awaitTermination()

print(f"Current Bronze rows: {spark.table(TABLE).count()}")

display(
    spark.table(TABLE)
    .groupBy("_delivery_id")
    .agg(F.count("*").alias("rows"))
    .orderBy("_delivery_id")
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

TABLE = "cat_adblh_dev.bronze.shipments_postgres_raw"

LANDING = (
    "abfss://landing@stadblhdeveus2001.dfs.core.windows.net/"
    "postgres/shipments"
)

# Keep the existing checkpoint. Do not reset processing progress.
CHECKPOINT = (
    "abfss://checkpoints@stadblhdeveus2001.dfs.core.windows.net/"
    "q2_shipments_directory_listing_v1"
)

SOURCE_COLUMNS = [
    "shipment_id",
    "order_id",
    "shipment_status",
    "updated_at",
    "source_version",
]

DEMO_IDS = [
    "q3_added_field_v1",
    "q3_bad_version_v1",
    "q3_bad_status_v1",
]

current = spark.table(TABLE)

# Reuse the actual types of the original five source fields.
SOURCE_SCHEMA = StructType([
    current.schema[name] for name in SOURCE_COLUMNS
])

# PROBLEM — An approved field must survive a fresh notebook session.
# carrier_code is now an accepted, optional source field.
# _rescued_data preserves future unexpected fields and incompatible values.
READ_SCHEMA = StructType(
    list(SOURCE_SCHEMA.fields)
    + [
        StructField("carrier_code", StringType(), True),
        StructField("_rescued_data", StringType(), True),
    ]
)

# Preserve the existing project source-system label.
labels = [
    row["_source_system"]
    for row in current.select("_source_system").distinct().collect()
]

if len(labels) != 1 or labels[0] is None:
    raise RuntimeError(
        "Expected one existing non-null source-system label."
    )

SOURCE_SYSTEM = labels[0]

# Add missing output columns without clearing existing records.
for name in [
    "carrier_code",
    "_rescued_data",
    "_containment_reason",
]:
    if name not in spark.table(TABLE).columns:
        spark.sql(
            f"ALTER TABLE {TABLE} ADD COLUMNS (`{name}` STRING)"
        )

# Retain the original learning-fixture setup.
# Existing fixture folders are skipped, so completed deliveries
# are not overwritten or intentionally recreated.
existing_folders = {
    item.name.rstrip("/")
    for item in dbutils.fs.ls(LANDING)
}

missing_demo_ids = [
    delivery_id
    for delivery_id in DEMO_IDS
    if f"delivery_id={delivery_id}" not in existing_folders
]

if missing_demo_ids:
    # Freeze one baseline row before deriving demonstration records.
    seed_rows = (
        spark.table(TABLE)
        .filter(F.col("_delivery_id") == "snapshot_baseline")
        .select(*SOURCE_COLUMNS)
        .orderBy("shipment_id")
        .limit(1)
        .collect()
    )

    if not seed_rows:
        raise RuntimeError(
            "No baseline row is available to derive the fixtures."
        )

    seed = spark.createDataFrame(seed_rows, SOURCE_SCHEMA)

    fixtures = {
        "q3_added_field_v1": (
            seed.withColumn("carrier_code", F.lit("UPS"))
        ),
        "q3_bad_version_v1": (
            seed.withColumn("source_version", F.lit("N/A"))
        ),
        "q3_bad_status_v1": (
            seed.withColumn(
                "shipment_status",
                F.lit("UNKNOWN_STATUS"),
            )
        ),
    }

    # Illustrative platform-created deliveries, not PostgreSQL extracts.
    # The folder check is suitable for this single-writer exercise.
    for delivery_id in missing_demo_ids:
        (
            fixtures[delivery_id]
            .coalesce(1)
            .write.mode("append")
            .parquet(
                f"{LANDING}/delivery_id={delivery_id}/"
            )
        )

print(
    "Approved reader schema initialized. "
    "Existing Bronze records, fixture files, and checkpoint retained."
)

In [0]:
from uuid import uuid4
from pyspark.sql import functions as F

before_count = spark.table(TABLE).count()
run_id = str(uuid4())

# PROBLEM — Unexpected fields and incompatible values must remain visible.
# Read carrier_code directly because it is now approved.
incoming = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.useManagedFileEvents", "false")
    .option("cloudFiles.useNotifications", "false")
    .option("cloudFiles.includeExistingFiles", "true")
    .option("cloudFiles.partitionColumns", "")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(READ_SCHEMA)
    .load(LANDING)
    .select(
        *SOURCE_COLUMNS,
        "carrier_code",
        "_rescued_data",
        F.col("_metadata.file_path").alias("_source_file"),
    )
)

classified = (
    incoming
    .withColumn("_source_system", F.lit(SOURCE_SYSTEM))
    .withColumn(
        "_delivery_id",
        F.regexp_extract(
            F.col("_source_file"),
            r"/delivery_id=([^/]+)/",
            1,
        ),
    )
    .withColumn("_run_id", F.lit(run_id))
    .withColumn("_ingested_at", F.current_timestamp())

    # PROBLEM — Schema compatibility does not establish business validity.
    # Assign the first matching reason using explicit learning rules.
    .withColumn(
        "_containment_reason",
        F.when(
            F.col("source_version").isNull()
            | (F.col("source_version") < 1),
            F.lit("INVALID_SOURCE_VERSION"),
        )
        .when(
            F.col("shipment_status").isNull()
            | ~F.col("shipment_status").isin(
                "delivered",
                "in_transit",
            ),
            F.lit("INVALID_SHIPMENT_STATUS"),
        )
        .when(
            F.col("_rescued_data").isNotNull(),
            F.lit("UNAPPROVED_FIELDS_OR_TYPES"),
        )
        .otherwise(F.lit(None).cast("string")),
    )
)

# PROBLEM — Normal restarts must preserve processing memory.
# Use the same checkpoint and Delta target.
query = (
    classified.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE)
)

# Wait for available input to finish processing before measuring output.
query.awaitTermination()

after_count = spark.table(TABLE).count()

print({
    "rows_before_ingestion": before_count,
    "rows_after_ingestion": after_count,
    "rows_added": after_count - before_count,
})

# Display current persisted fixture results.
display(
    spark.table(TABLE)
    .filter(F.col("_delivery_id").isin(DEMO_IDS))
    .select(
        "_delivery_id",
        "carrier_code",
        "source_version",
        "shipment_status",
        "_rescued_data",
        "_containment_reason",
    )
    .orderBy("_delivery_id")
)

In [0]:
from pyspark.sql import functions as F

# Explicit simulated approval for this learning exercise:
# carrier_code is approved as a string field.
if "carrier_code" not in spark.table(TABLE).columns:
    spark.sql(f"ALTER TABLE {TABLE} ADD COLUMNS (carrier_code STRING)")

ADDITION_PATH = f"{LANDING}/delivery_id=q3_added_field_v1/"

# PROBLEM — Reprocessing must not append a second copy.
# Reread only the affected delivery as a batch, then update its rows.
approved_source = (
    spark.read.parquet(ADDITION_PATH)
    .select(
        "shipment_id",
        "carrier_code",
        F.col("_metadata.file_path").alias("_source_file"),
    )
)
approved_source.createOrReplaceTempView("q3_approved_carrier")

spark.sql(f"""
    MERGE INTO {TABLE} AS target
    USING q3_approved_carrier AS source
    ON target._delivery_id = 'q3_added_field_v1'
       AND target.shipment_id = source.shipment_id
       AND target._source_file = source._source_file
    WHEN MATCHED
         AND target._containment_reason = 'UNAPPROVED_FIELDS_OR_TYPES'
    THEN UPDATE SET
        target.carrier_code = source.carrier_code,
        target._containment_reason = NULL
""")

# Update the notebook's expected reader schema for future executions.
# Cell 5 will use this updated schema when rerun in this session.
if "carrier_code" not in READ_SCHEMA.fieldNames():
    READ_SCHEMA = READ_SCHEMA.add("carrier_code", "string", True)

# Keep a complete source selection, including the approved addition.
# Apply the cell-5 reader adjustment explained below before future arrivals.

before_lookup = {
    row["_delivery_id"]: row["_containment_reason"]
    for row in before_approval
}

final_rows = (
    spark.table(TABLE)
    .filter(F.col("_delivery_id").isin(DEMO_IDS))
    .select(
        "_delivery_id",
        "carrier_code",
        "source_version",
        "shipment_status",
        "_rescued_data",
        "_containment_reason",
    )
    .orderBy("_delivery_id")
)

display(final_rows)

final_count = spark.table(TABLE).count()
print({
    "classification_before_approval": before_lookup,
    "rows_after_ingestion": after_count,
    "rows_after_approved_update": final_count,
    "rows_added_by_approved_update": final_count - after_count,
})

# One reconciliation for this exercise, using the persisted result.
counts = {
    r["_delivery_id"]: r["count"]
    for r in (
        spark.table(TABLE)
        .filter(F.col("_delivery_id").isin(DEMO_IDS))
        .groupBy("_delivery_id")
        .count()
        .collect()
    )
}
if counts != {delivery_id: 1 for delivery_id in DEMO_IDS}:
    raise RuntimeError(f"Unexpected fixture reconciliation: {counts}")

if final_count != after_count:
    raise RuntimeError("The approved update unexpectedly changed row count.")

print("Q3 fixture reconciliation: PASS")